In [0]:
# ============================================================
# NOTEBOOK 1 — BRONZE LAYER (Raw Ingestion)
# Project : E-commerce Sales Analytics
# Compute : Databricks Serverless
# Storage : Unity Catalog Volumes
# ============================================================

# --- Source path (your uploaded files) ---
VOLUME_PATH = "/Volumes/brazilian-ecommerce/filestore/olis/"

# --- Bronze Delta output path (Unity Catalog managed) ---
BRONZE_CATALOG  = "brazilian-ecommerce"
BRONZE_SCHEMA   = "bronze"

# Create bronze schema/database if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`")
print(f"Schema ready: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
print(f"Source path : {VOLUME_PATH}")
print(f"Spark version: {spark.version}")

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType
)

orders_schema = StructType([
    StructField("order_id",                        StringType(),  True),
    StructField("customer_id",                     StringType(),  True),
    StructField("order_status",                    StringType(),  True),
    StructField("order_purchase_timestamp",        StringType(),  True),
    StructField("order_approved_at",               StringType(),  True),
    StructField("order_delivered_carrier_date",    StringType(),  True),
    StructField("order_delivered_customer_date",   StringType(),  True),
    StructField("order_estimated_delivery_date",   StringType(),  True)
])

order_items_schema = StructType([
    StructField("order_id",            StringType(),   True),
    StructField("order_item_id",       IntegerType(),  True),
    StructField("product_id",          StringType(),   True),
    StructField("seller_id",           StringType(),   True),
    StructField("shipping_limit_date", StringType(),   True),
    StructField("price",               DoubleType(),   True),
    StructField("freight_value",       DoubleType(),   True)
])

customers_schema = StructType([
    StructField("customer_id",              StringType(), True),
    StructField("customer_unique_id",       StringType(), True),
    StructField("customer_zip_code_prefix", StringType(), True),
    StructField("customer_city",            StringType(), True),
    StructField("customer_state",           StringType(), True)
])

products_schema = StructType([
    StructField("product_id",                   StringType(),  True),
    StructField("product_category_name",         StringType(),  True),
    StructField("product_name_lenght",           IntegerType(), True),
    StructField("product_description_lenght",    IntegerType(), True),
    StructField("product_photos_qty",            IntegerType(), True),
    StructField("product_weight_g",              DoubleType(),  True),
    StructField("product_length_cm",             DoubleType(),  True),
    StructField("product_height_cm",             DoubleType(),  True),
    StructField("product_width_cm",              DoubleType(),  True)
])

payments_schema = StructType([
    StructField("order_id",              StringType(),  True),
    StructField("payment_sequential",    IntegerType(), True),
    StructField("payment_type",          StringType(),  True),
    StructField("payment_installments",  IntegerType(), True),
    StructField("payment_value",         DoubleType(),  True)
])

print("All schemas defined.")

In [0]:
from pyspark.sql.functions import current_timestamp, lit

def read_csv(filename, schema):
    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("nullValue", "NA")
        .option("emptyValue", None)
        .schema(schema)
        .load(VOLUME_PATH + filename)
    )

df_orders    = read_csv("olist_orders_dataset.csv",        orders_schema)
df_items     = read_csv("olist_order_items_dataset.csv",   order_items_schema)
df_customers = read_csv("olist_customers_dataset.csv",     customers_schema)
df_products  = read_csv("olist_products_dataset.csv",      products_schema)
df_payments  = read_csv("olist_order_payments_dataset.csv",payments_schema)

# Count rows — this triggers the first Spark action (actual data read happens here)
counts = {
    "orders":    df_orders.count(),
    "items":     df_items.count(),
    "customers": df_customers.count(),
    "products":  df_products.count(),
    "payments":  df_payments.count(),
}
for name, cnt in counts.items():
    print(f"  {name:12} → {cnt:,} rows")

In [0]:
def add_audit_columns(df, source_name):
    """
    Add standard lineage columns to every Bronze table.
    In production these are mandatory — they tell you WHEN
    and WHERE each row came from.
    """
    return (
        df
        .withColumn("_ingested_at",  current_timestamp())
        .withColumn("_source_file",  lit(source_name))
        .withColumn("_layer",        lit("bronze"))
    )

df_orders_b    = add_audit_columns(df_orders,    "olist_orders_dataset.csv")
df_items_b     = add_audit_columns(df_items,     "olist_order_items_dataset.csv")
df_customers_b = add_audit_columns(df_customers, "olist_customers_dataset.csv")
df_products_b  = add_audit_columns(df_products,  "olist_products_dataset.csv")
df_payments_b  = add_audit_columns(df_payments,  "olist_order_payments_dataset.csv")

print("Audit columns added to all DataFrames.")
df_orders_b.printSchema()

In [0]:
tables_to_write = [
    ("orders",       df_orders_b),
    ("order_items",  df_items_b),
    ("customers",    df_customers_b),
    ("products",     df_products_b),
    ("payments",     df_payments_b),
]

for table_name, df in tables_to_write:
    full_table = f"`{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`{table_name}`"
    
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(full_table)
    )
    print(f"  ✓ Written → {full_table}")

print("\nAll Bronze tables written successfully!")

In [0]:
# List all Bronze tables
spark.sql(f"SHOW TABLES IN `{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`").show()

In [0]:
# Row counts from actual Delta tables
for table_name, _ in tables_to_write:
    full = f"`{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`{table_name}`"
    cnt = spark.sql(f"SELECT COUNT(*) as cnt FROM {full}").collect()[0]["cnt"]
    print(f"  {table_name:15} → {cnt:,} rows")

In [0]:
# THE most important cell for interviews — Delta history
spark.sql(f"""
    DESCRIBE HISTORY `{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`orders`
""").select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

In [0]:
# Time travel — read version 0 (your original load)
df_v0 = (spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(f"`{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`orders`")
)
print(f"Version 0 row count: {df_v0.count():,}")

In [0]:
# Always preview Bronze data — catch ingestion issues early
print("=== ORDERS sample ===")
display(spark.sql(f"""
    SELECT order_id, customer_id, order_status, 
           order_purchase_timestamp, _ingested_at, _source_file
    FROM `{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`orders`
    LIMIT 5
"""))

In [0]:
print("=== ORDER STATUS distribution ===")
display(spark.sql(f"""
    SELECT order_status, COUNT(*) as count
    FROM `{BRONZE_CATALOG}`.`{BRONZE_SCHEMA}`.`orders`
    GROUP BY order_status
    ORDER BY count DESC
"""))